In [1]:
import sys
import os
from pyspark.sql.window import Window
from pyspark.sql import SparkSession
from pyspark.sql.column import Column
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

In [2]:
APP_NAME = "Discharge App - Gold"

In [3]:
read_csv = lambda path: spark.read.option("header", "true").option("sep", ";").csv(path)
DATA_PATH = os.getenv("DATA_PATH")

## Dimension

In [4]:
dim_cause_of_death_df = read_csv(f"{DATA_PATH}/raw/dimensions/dim_cause_of_death.csv")\
    .select(
        F.col("id"),
        F.col("subcategory_code")
    )

dim_cause_of_death_df.show()

+---+----------------+
| id|subcategory_code|
+---+----------------+
|  1|            A000|
|  2|            A001|
|  3|            A009|
|  4|            A010|
|  5|            A011|
|  6|            A012|
|  7|            A013|
|  8|            A014|
|  9|            A020|
| 10|            A021|
| 11|            A022|
| 12|            A028|
| 13|            A029|
| 14|            A030|
| 15|            A031|
| 16|            A032|
| 17|            A033|
| 18|            A038|
| 19|            A039|
| 20|            A040|
+---+----------------+
only showing top 20 rows



In [7]:
dim_cause_of_death_df.filter((F.col("subcategory_code").isNull()) | (F.col("subcategory_code") == "")).show()

+---+----------------+
| id|subcategory_code|
+---+----------------+
+---+----------------+



## Fact table (Raw)

In [23]:
schema = StructType([
    StructField("D_R", IntegerType(), True),
    StructField("ser_salud", IntegerType(), True),
    StructField("estab", IntegerType(), True),
    StructField("sexo", IntegerType(), True),
    StructField("edad", IntegerType(), True),
    StructField("previ", IntegerType(), True),
    StructField("benef", IntegerType(), True),
    StructField("comuna", IntegerType(), True),
    StructField("fecha_egre", StringType(), True),
    StructField("sercle", StringType(), True),
    StructField("dias_estad", IntegerType(), True),
    StructField("diag1", StringType(), True),
    StructField("diag2", StringType(), True),
    StructField("cond_egr", IntegerType(), True),
    StructField("intervq", IntegerType(), True),
    StructField("region", IntegerType(), True),
    StructField("serv_res", StringType(), True),
])

raw_discharges = spark.read\
    .format('csv')\
    .option('header', 'true')\
    .option('sep', ';')\
    .schema(schema)\
    .load(f"{DATA_PATH}/raw/discharges/{{2001,2002,2003,2004,2006}}.csv")

raw_discharges.show()

+----+---------+-----+----+----+-----+-----+------+----------+------+----------+-----+-----+--------+-------+------+--------+
| D_R|ser_salud|estab|sexo|edad|previ|benef|comuna|fecha_egre|sercle|dias_estad|diag1|diag2|cond_egr|intervq|region|serv_res|
+----+---------+-----+----+----+-----+-----+------+----------+------+----------+-----+-----+--------+-------+------+--------+
|null|        1|  100|   2|  37|    1|    1|  1101|  7/8/2001|600-02|         1| O039|     |       1|      1|     1|       2|
|null|        1|  100|   1|  33|    1|    1|  1101| 8/27/2001|100-99|        16| C629|     |       1|      2|     1|       2|
|null|        1|  100|   1|   6|    1|    1|  1101| 5/24/2001|500-99|         1| A09X|     |       1|      2|     1|       2|
|null|        1|  100|   2|  47|    1|    1|  1101|  4/4/2001|200-00|        25| K810|     |       1|      1|     1|       2|
|null|        1|  100|   1|   0|    4| null|  1101|12/24/2001|500-99|         2| S411| V439|       1|      2|     1|  

In [30]:
raw_discharges.filter(F.col("diag2").startswith("U802")).show(100)

[Stage 56:>                                                         (0 + 5) / 5]

+----+---------+-----+----+----+-----+-----+------+----------+------+----------+-----+-----+--------+-------+------+--------+
| D_R|ser_salud|estab|sexo|edad|previ|benef|comuna|fecha_egre|sercle|dias_estad|diag1|diag2|cond_egr|intervq|region|serv_res|
+----+---------+-----+----+----+-----+-----+------+----------+------+----------+-----+-----+--------+-------+------+--------+
|null|        9|  101|   2|   9|    1|    1| 13302| 3/22/2001|400-03|         3| S424| U802|       1|      1|    13|       9|
+----+---------+-----+----+----+-----+-----+------+----------+------+----------+-----+-----+--------+-------+------+--------+



In [33]:
raw_discharges.filter(F.col("diag2").rlike("^[a-zA-Z]{2}")).show(100)

[Stage 60:=======================>                                  (2 + 3) / 5]

+----+---------+-----+----+----+-----+-----+------+----------+------+----------+-----+-------+--------+-------+------+--------+
| D_R|ser_salud|estab|sexo|edad|previ|benef|comuna|fecha_egre|sercle|dias_estad|diag1|  diag2|cond_egr|intervq|region|serv_res|
+----+---------+-----+----+----+-----+-----+------+----------+------+----------+-----+-------+--------+-------+------+--------+
|null|        3|  102|   1|  45|    1|    1|  2301|21-10-2002|   999|         1| T140|   WO1O|       1|      2|     2|       3|
|null|       11|  100|   1|   6|    1|    4|  7101|11/18/2003|   142|        11| T290|   XO99|       1|      1|     7|      16|
|null|       13|  190|   1|  41|    1|    3| 13401|  8/8/2003|   120|        38| S750|   XP34|       1|      1|    13|      13|
|null|        5|  101|   1|  11|    1|    1|  4102|11-11-2004|   150|        16| T749|XY07.X.|       1|      2|     4|       5|
|null|        7|  100|   1|   1|    1|    3|  5109|02-02-2004|   140|         1| T310|  XO9.9|       1| 

## Fact table (Silver uniformed)

In [25]:
cols_to_load = [
    "date", "gender", "age", "health_insurance", "city", 
    "healthcare_service", "facility", "cause_of_death_1", "cause_of_death_2"
]

silver_uniformed_df = spark.read.parquet(f"{DATA_PATH}/silver/uniformed_discharges").select(*cols_to_load)
silver_uniformed_df.show()

[Stage 41:===========================================>              (3 + 1) / 4]

+----------+------+---+----------------+----+------------------+--------+----------------+----------------+
|      date|gender|age|health_insurance|city|healthcare_service|facility|cause_of_death_1|cause_of_death_2|
+----------+------+---+----------------+----+------------------+--------+----------------+----------------+
|01-01-2006|     1| 49|               1|1201|                 1|     100|            C951|                |
|01-01-2006|     1|  4|               1|1201|                 1|     100|            T751|            W679|
|01-01-2006|     2| 21|               1|1201|                 1|     100|            O809|                |
|01-01-2006|     2| 30|               1|1201|                 1|     100|            N940|                |
|01-01-2006|     1| 76|               1|1201|                 1|     100|            I619|                |
|02-01-2006|     1| 46|               1|1201|                 1|     100|            I509|                |
|02-01-2006|     2| 76|     

In [26]:
silver_uniformed_df.filter(F.col("cause_of_death_2").startswith("WW")).show(100)

+----+------+---+----------------+----+------------------+--------+----------------+----------------+
|date|gender|age|health_insurance|city|healthcare_service|facility|cause_of_death_1|cause_of_death_2|
+----+------+---+----------------+----+------------------+--------+----------------+----------------+
+----+------+---+----------------+----+------------------+--------+----------------+----------------+



## Fact table (Silver cleaned)

In [27]:
windowSpec = Window.orderBy(F.monotonically_increasing_id())

discharges_df = spark\
    .read\
    .parquet(f"{DATA_PATH}/silver/cleaned_discharges")\
    .withColumn("id", F.row_number().over(windowSpec))\
    .select("id", "year","gender", "health_insurance", "city", "healthcare_service", "cause_of_death_1", "cause_of_death_2")

discharges_df.show()

26/05/05 16:46:51 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/05 16:46:51 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/05 16:46:51 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
[Stage 46:==================================================>       (7 + 1) / 8]

+---+----+------+----------------+----+------------------+----------------+----------------+
| id|year|gender|health_insurance|city|healthcare_service|cause_of_death_1|cause_of_death_2|
+---+----+------+----------------+----+------------------+----------------+----------------+
|  1|2001|     2|               1|1101|                 1|            O039|             N/E|
|  2|2001|     1|               1|5108|                 7|            M139|             N/E|
|  3|2001|     1|               1|1101|                 1|            C629|             N/E|
|  4|2001|     2|               1|5108|                 7|            O809|             N/E|
|  5|2001|     1|               1|1101|                 1|            A09X|             N/E|
|  6|2001|     2|               1|5108|                 7|            O809|             N/E|
|  7|2001|     2|               1|1101|                 1|            K810|             N/E|
|  8|2001|     2|               1|5108|                 7|            

In [28]:
discharges_df.filter(F.col("cause_of_death_2").startswith("WW")).show(100)

26/05/05 16:47:03 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/05 16:47:03 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/05 16:47:03 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/05 16:47:14 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/05 16:47:14 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
[Stage 50:>                                                         (0 + 1) / 1]

+---+----+------+----------------+----+------------------+----------------+----------------+
| id|year|gender|health_insurance|city|healthcare_service|cause_of_death_1|cause_of_death_2|
+---+----+------+----------------+----+------------------+----------------+----------------+
+---+----+------+----------------+----+------------------+----------------+----------------+



## Fact table (Gold)

In [7]:
windowSpec = Window.orderBy(F.monotonically_increasing_id())

gold_df = spark\
    .read\
    .option("header", "true") \
    .csv(f"{DATA_PATH}/gold/discharges/part-*")\
    .withColumn("id", F.row_number().over(windowSpec))\
    .select(
        "id", 
        "health_insurance_id",
        "year_id", 
        "gender_id", 
        "age_id", 
        "city_id", 
        "healthcare_service_id", 
        "main_cause_of_death_id",
        "external_cause_of_death_id"
    )

gold_df.show()

26/05/05 21:36:10 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/05 21:36:10 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/05 21:36:10 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
[Stage 9:====================================>                      (5 + 3) / 8]

+---+-------------------+-------+---------+------+-------+---------------------+----------------------+--------------------------+
| id|health_insurance_id|year_id|gender_id|age_id|city_id|healthcare_service_id|main_cause_of_death_id|external_cause_of_death_id|
+---+-------------------+-------+---------+------+-------+---------------------+----------------------+--------------------------+
|  1|                  1|     12|        2|    46|    248|                    7|                  5635|                    999999|
|  2|                  1|     12|        1|    28|    722|                   32|                  4699|                    999999|
|  3|                  1|     12|        1|    50|    248|                    7|                  1055|                    999999|
|  4|                  1|     12|        2|    53|    722|                   32|                  5954|                    999999|
|  5|                  1|     12|        1|    77|    248|                    7|   

In [8]:
gold_df.filter(F.col("external_cause_of_death_id").rlike("^[a-zA-Z]")).count()

0

In [9]:
gold_df.filter(F.col("external_cause_of_death_id").rlike("^[a-zA-Z]")).groupBy("year_id").agg(F.count("year_id").alias("count")).show()

[Stage 14:=======>                                                  (1 + 7) / 8]

+-------+-----+
|year_id|count|
+-------+-----+
+-------+-----+



[Stage 14:=====================>                                    (3 + 5) / 8]

In [10]:
gold_df.filter(F.col("main_cause_of_death_id").rlike("^[a-zA-Z]")).count()

0

In [11]:
gold_df.filter(F.col("main_cause_of_death_id").rlike("^[a-zA-Z]")).groupBy("year_id").agg(F.count("year_id").alias("count")).show()

[Stage 18:===========================================>              (6 + 2) / 8]

+-------+-----+
|year_id|count|
+-------+-----+
+-------+-----+



In [13]:
gold_df\
    .filter(
        (F.col("external_cause_of_death_id") == 999999)
    )\
    .groupBy("year_id")\
    .count()\
    .sort(F.col("year_id").asc())\
    .show()

[Stage 19:==================================================>       (7 + 1) / 8]

+-------+-------+
|year_id|  count|
+-------+-------+
|     12|1398302|
|     13|1415444|
|     14|1450058|
|     15|1455219|
|     16|1474561|
|     17|1479707|
|     18|1480094|
|     19|1448876|
|     20|1525410|
|     21|1471099|
|     22|1495269|
|     23|1504065|
|     24|1523005|
|     25|1508883|
|     26|1517576|
|     27|1485188|
|     28|1482586|
|     29|1515073|
+-------+-------+

